# DecodeLabs — Project 1: Data Cleaning & Preparation
**Batch 2026** | Tool: Python (pandas)

### Dataset: Cafe Sales — Dirty Data
- Source: Kaggle (cafe-sales-dirty-data-for-cleaning-training)
- 315 raw rows × 8 columns
- Issues: missing values, duplicates, bad date formats, inconsistent text, outliers, negative values


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('cafe_sales_DIRTY.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (315, 8)


,Transaction_ID,Date,Item,Category,Quantity,Price_Per_Unit,Total_Spent,Payment_Method
0,TXN0051,2023-02-04,Coffee,Hot Drinks,2.0,3.5,7.0,Cash
1,TXN0299,2023-07-28,Latte,Hot Drinks,5.0,4.5,22.5,Debit Card
2,TXN0252,2023-10-30,Sandwich,Food,5.0,6.0,30.0,Debit Card
3,TXN0232,2023-07-05,Cake,Food,5.0,4.5,22.5,Mobile Payment
4,TXN0016,2023-08-21,Muffin,Food,3.0,3.5,10.5,Debit Card


In [2]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nData types:")
print(df.dtypes)

Missing values per column:
Transaction_ID     0
Date               0
Item               0
Category           5
Quantity          16
Price_Per_Unit    15
Total_Spent       25
Payment_Method    20
dtype: int64

Duplicate rows: 8

Data types:
Transaction_ID     object
Date               object
Item               object
Category           object
Quantity          float64
Price_Per_Unit    float64
Total_Spent       float64
Payment_Method     object
dtype: object


In [3]:
# Check inconsistent text values
print("Unique Item values:", df['Item'].unique())
print("\nUnique Payment_Method values:", df['Payment_Method'].unique())

Unique Item values: ['Coffee' 'Latte' 'Sandwich' 'Cake' 'Muffin' 'Espresso' 'Juice' 'Salad'
 ' Coffee' 'Bagel' 'Tea' 'TEA' 'Coffee ' 'sandwich' 'tea' 'LATTE' 'latte'
 ' tea ' ' Latte' 'COFFEE' 'Sandwich ' 'SANDWICH' 'coffee']

Unique Payment_Method values: ['Cash' 'Debit Card' 'Mobile Payment' 'debit card' 'Credit Card' nan
 'CREDIT CARD' 'credit card' 'Mobile payment' 'CASH' 'mobile payment'
 'cash']


In [4]:
# Check date format issues
bad_dates = df[df['Date'].str.contains('/', na=False)]
print(f"Bad date format rows: {len(bad_dates)}")
bad_dates[['Transaction_ID','Date']].head(5)

Bad date format rows: 20


,Transaction_ID,Date
31,TXN0085,06/12/2023
80,TXN0091,02/01/2023
84,TXN0281,01/06/2023
92,TXN0175,23/05/2023
94,TXN0074,19/10/2023


## Phase 1: Strategic Imputation
Handle missing values — don't just delete (listwise deletion reduces statistical power).
- **Numeric skewed** → Median
- **Numeric normal** → Mean  
- **Categorical** → Mode

In [5]:
# CR001: Standardize all dates to ISO 8601 (YYYY-MM-DD)
def standardize_date(val):
    if pd.isna(val):
        return np.nan
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y'):
        try:
            return pd.to_datetime(val, format=fmt).strftime('%Y-%m-%d')
        except:
            continue
    return np.nan

bad_dates_count = df['Date'].str.contains('/', na=False).sum()
df['Date'] = df['Date'].apply(standardize_date)
print(f"[CR001] Fixed {bad_dates_count} incorrectly formatted dates → YYYY-MM-DD")

[CR001] Fixed 20 incorrectly formatted dates → YYYY-MM-DD


In [6]:
# CR002 & CR003: Quantity — fill nulls with median, fix negatives
median_qty = int(df[df['Quantity'] > 0]['Quantity'].median())
null_qty = df['Quantity'].isnull().sum()
neg_qty = (df['Quantity'] < 0).sum()

df['Quantity'] = df['Quantity'].fillna(median_qty)
df.loc[df['Quantity'] < 0, 'Quantity'] = median_qty

print(f"[CR002] Filled {null_qty} missing Quantity values with median = {median_qty}")
print(f"[CR003] Replaced {neg_qty} negative Quantity values with median = {median_qty}")

[CR002] Filled 16 missing Quantity values with median = 3
[CR003] Replaced 5 negative Quantity values with median = 3


In [7]:
# CR004: Price_Per_Unit — cap outliers, then impute nulls with per-item median
outliers = (df['Price_Per_Unit'] > 50).sum()
df.loc[df['Price_Per_Unit'] > 50, 'Price_Per_Unit'] = np.nan
null_price = df['Price_Per_Unit'].isnull().sum()

item_median_price = df.groupby('Item')['Price_Per_Unit'].median()
df['Price_Per_Unit'] = df.apply(
    lambda r: item_median_price.get(r['Item'], df['Price_Per_Unit'].median())
    if pd.isna(r['Price_Per_Unit']) else r['Price_Per_Unit'], axis=1
)
print(f"[CR004] Capped {outliers} price outliers (>$50), imputed {null_price} nulls with per-item median")

[CR004] Capped 3 price outliers (>$50), imputed 18 nulls with per-item median


In [8]:
# CR004: Price_Per_Unit — cap outliers, then impute nulls with per-item median
outliers = (df['Price_Per_Unit'] > 50).sum()
df.loc[df['Price_Per_Unit'] > 50, 'Price_Per_Unit'] = np.nan
null_price = df['Price_Per_Unit'].isnull().sum()

item_median_price = df.groupby('Item')['Price_Per_Unit'].median()
df['Price_Per_Unit'] = df.apply(
    lambda r: item_median_price.get(r['Item'], df['Price_Per_Unit'].median())
    if pd.isna(r['Price_Per_Unit']) else r['Price_Per_Unit'], axis=1
)
print(f"[CR004] Capped {outliers} price outliers (>$50), imputed {null_price} nulls with per-item median")

[CR004] Capped 0 price outliers (>$50), imputed 0 nulls with per-item median


In [11]:
# CR005: Payment_Method — fill nulls with mode (most frequent value)
mode_pay = df['Payment_Method'].dropna().mode()[0]
null_pay = df['Payment_Method'].isnull().sum()
df['Payment_Method'] = df['Payment_Method'].fillna(mode_pay)
print(f"[CR005] Filled {null_pay} missing Payment_Method values with mode = '{mode_pay}'")

[CR005] Filled 20 missing Payment_Method values with mode = 'Debit Card'


## Phase 2: The Integrity Audit
One Truth, One Record — eliminate all duplicates.

In [12]:
# CR006: Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset='Transaction_ID', keep='first')
removed = before - len(df)
print(f"[CR006] Removed {removed} duplicate rows")
print(f"Remaining: {len(df)} unique records")

# Verify
assert df.duplicated().sum() == 0
print(f"Duplicate check: {df.duplicated().sum()} ✓")

[CR006] Removed 15 duplicate rows
Remaining: 300 unique records
Duplicate check: 0 ✓


## Phase 3: Speak One Language
Standardize all text, formats, and data types.

In [13]:
# CR007: Standardize Item names — strip whitespace + title case
before_unique = df['Item'].nunique()
df['Item'] = df['Item'].str.strip().str.title()
after_unique = df['Item'].nunique()
print(f"[CR007] Item unique values: {before_unique} messy variants → {after_unique} clean values")
print("Clean values:", sorted(df['Item'].unique()))

[CR007] Item unique values: 23 messy variants → 10 clean values
Clean values: ['Bagel', 'Cake', 'Coffee', 'Espresso', 'Juice', 'Latte', 'Muffin', 'Salad', 'Sandwich', 'Tea']


In [14]:
# CR008: Standardize Payment_Method casing
df['Payment_Method'] = df['Payment_Method'].str.strip().str.title()
print(f"[CR008] Payment_Method values: {sorted(df['Payment_Method'].unique())}")

[CR008] Payment_Method values: ['Cash', 'Credit Card', 'Debit Card', 'Mobile Payment']


In [15]:
# CR009: Fix 'NULL' string Category values using Item lookup
item_category = {
    'Coffee': 'Hot Drinks', 'Tea': 'Hot Drinks', 'Latte': 'Hot Drinks', 'Espresso': 'Hot Drinks',
    'Sandwich': 'Food', 'Cake': 'Food', 'Salad': 'Food', 'Muffin': 'Food', 'Bagel': 'Food',
    'Juice': 'Cold Drinks'
}
df['Category'] = df['Category'].replace('NULL', np.nan)
df['Category'] = df.apply(
    lambda r: item_category.get(r['Item'], 'Unknown') if pd.isna(r['Category']) else r['Category'], axis=1
)
print(f"[CR009] Category fixed — unique values: {sorted(df['Category'].unique())}")

[CR009] Category fixed — unique values: ['Cold Drinks', 'Food', 'Hot Drinks']


In [16]:
# CR010: Recalculate Total_Spent = Price_Per_Unit x Quantity, 2 decimal places
df['Total_Spent'] = (df['Price_Per_Unit'] * df['Quantity']).round(2)
print("[CR010] Total_Spent recalculated and rounded to 2dp for all rows")

[CR010] Total_Spent recalculated and rounded to 2dp for all rows


In [17]:
# CR011: Enforce correct data types
df['Date'] = pd.to_datetime(df['Date'])
df['Quantity'] = df['Quantity'].astype(int)
df['Price_Per_Unit'] = df['Price_Per_Unit'].round(2)
print("[CR011] Data types enforced:")
print(df.dtypes)

[CR011] Data types enforced:
Transaction_ID            object
Date              datetime64[ns]
Item                      object
Category                  object
Quantity                   int64
Price_Per_Unit           float64
Total_Spent              float64
Payment_Method            object
dtype: object


## Final Verification — 0% Error Gate
Must prove: zero duplicate IDs and zero incorrectly formatted dates.

In [19]:
# 0% Error Rate Verification
checks = {
    'Duplicate rows':      df.duplicated().sum(),
    'Missing values':      df.isnull().sum().sum(),
    'Negative quantities': (df['Quantity'] < 0).sum(),
    'Bad date formats':    df['Date'].isnull().sum(),
}

print("=== FINAL VERIFICATION ===")
all_passed = True
for check, result in checks.items():
    status = '✓ PASS' if result == 0 else '✗ FAIL'
    if result != 0: all_passed = False
    print(f"  {check:<25} {result}  {status}")

print(f"\nFinal shape: {df.shape}")
print(f"\n{'ALL CHECKS PASSED ' if all_passed else '❌ Issues remain'}")

=== FINAL VERIFICATION ===
  Duplicate rows            0  ✓ PASS
  Missing values            0  ✓ PASS
  Negative quantities       0  ✓ PASS
  Bad date formats          0  ✓ PASS

Final shape: (300, 8)

ALL CHECKS PASSED 
